<a href="https://colab.research.google.com/github/raizelk/AI-powered-Fintech-platform/blob/main/movie_review_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jcblaise/imdb-sentiments")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-sentiments' dataset.
Path to dataset files: /kaggle/input/imdb-sentiments


In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing

In [ ]:
df = pd.read_csv("/root/.cache/kagglehub/datasets/jcblaise/imdb-sentiments/versions/3/train.csv")


In [ ]:
df.head()

,text,sentiment
0,For a movie that gets no respect there sure ar...,0
1,Bizarre horror movie filled with famous faces ...,0
2,"A solid, if unremarkable film. Matthau, as Ein...",0
3,It's a strange feeling to sit alone in a theat...,0
4,"You probably all already know this by now, but...",0


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)   # remove punctuation/numbers
    return text

In [ ]:
df['cleaned'] = df['text'].apply(clean_text)

In [ ]:
# TOKENIZATION
df['tokens'] = df['cleaned'].apply(word_tokenize)

In [ ]:
stop_words = set(stopwords.words('english'))
df['no_stopwords'] = df['tokens'].apply(lambda x: [w for w in x if w not in stop_words])

In [ ]:
df.head()

In [ ]:
# STEMMING
stemmer = PorterStemmer()
df['stemmed'] = df['no_stopwords'].apply(lambda x: [stemmer.stem(w) for w in x])

In [ ]:
# LEMMATIZATION
lemmatizer = WordNetLemmatizer()
df['lemmatized'] = df['no_stopwords'].apply(lambda x: [lemmatizer.lemmatize(w) for w in x])

In [ ]:
df.head()

In [ ]:
import pickle
import os
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

OUT_DIR = "./models"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Convert list → string
df["stemmed_str"] = df["stemmed"].apply(lambda x: " ".join(x))
df["lemmatized_str"] = df["lemmatized"].apply(lambda x: " ".join(x))

MAX_FEATURES = 5000  # ***** ADD THIS *****

cv = CountVectorizer(max_features=MAX_FEATURES)
tfv = TfidfVectorizer(max_features=MAX_FEATURES)

X_bow = cv.fit_transform(df["stemmed_str"])
X_tfidf = tfv.fit_transform(df["lemmatized_str"])

# Save vectorizers
pickle.dump(cv, open(os.path.join(OUT_DIR, "count_vectorizer.pkl"), "wb"))
pickle.dump(tfv, open(os.path.join(OUT_DIR, "tfidf_vectorizer.pkl"), "wb"))



In [ ]:
#Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

# If sentiment is 'negative'/'positive', convert to 0/1
if df['sentiment'].dtype == object:
    df['label'] = df['sentiment'].map({'negative':0,'positive':1})
else:
    df['label'] = df['sentiment']  # already numeric

df = df[df['tokens'].map(lambda x: len(x)>0)].reset_index(drop=True)
print("Samples:", len(df))
print("Sample tokens:", df.loc[0,'tokens'][:15])


!pip install -q gensim

from gensim.models import Word2Vec

sentences = df['tokens'].tolist()

VECTOR_SIZE = 100   # embedding dimension
WINDOW = 5
MIN_COUNT = 3       # min freq to keep a word
EPOCHS = 10
WORKERS = 4

# CBOW (sg=0)
w2v_cbow = Word2Vec(
    sentences=sentences,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    sg=0,
    workers=WORKERS,
    epochs=EPOCHS
)
w2v_cbow.save("w2v_cbow.model")

# Skip-gram (sg=1)
w2v_sg = Word2Vec(
    sentences=sentences,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    sg=1,
    workers=WORKERS,
    epochs=EPOCHS
)
w2v_sg.save("w2v_sg.model")

print("CBOW vocab size:", len(w2v_cbow.wv))
print("Skip-gram vocab size:", len(w2v_sg.wv))
# Example neighbors (may fail if word not in vocab)
for word in ['movie','good','bad']:
    if word in w2v_sg.wv.key_to_index:
        print("SG neighbors for", word, ":", w2v_sg.wv.most_similar(word, topn=6))
    if word in w2v_cbow.wv.key_to_index:
        print("CBOW neighbors for", word, ":", w2v_cbow.wv.most_similar(word, topn=6))


In [ ]:
#GloVe
!wget -q --show-progress http://nlp.stanford.edu/data/glove.6B.zip -O glove.6B.zip
!unzip -q -o glove.6B.zip glove.6B.100d.txt
from gensim.scripts.glove2word2vec import glove2word2vec
from gensim.models import KeyedVectors

glove2word2vec("glove.6B.100d.txt", "glove.6B.100d.word2vec.txt")
glove_kv = KeyedVectors.load_word2vec_format("glove.6B.100d.word2vec.txt")

print("Loaded GloVe! Vocab size:", len(glove_kv))
glove_kv.most_similar("movie", topn=5)

!unzip -q -o glove.6B.zip glove.6B.100d.txt
import os
print("Exists:", os.path.exists("glove.6B.100d.txt"))

from gensim.scripts.glove2word2vec import glove2word2vec

glove2word2vec("glove.6B.100d.txt", "glove.6B.100d.word2vec.txt")
from gensim.models import KeyedVectors

glove_kv = KeyedVectors.load_word2vec_format("glove.6B.100d.word2vec.txt", binary=False)

print("Loaded GloVe! Vocabulary size =", len(glove_kv))
import numpy as np

def doc_vec_glove(tokens):
    vecs = [glove_kv[w] for w in tokens if w in glove_kv.key_to_index]
    if len(vecs) == 0:
        return np.zeros(glove_kv.vector_size)
    return np.mean(vecs, axis=0)
X_glove = np.vstack([doc_vec_glove(t) for t in df['tokens']])

# labels
if df['sentiment'].dtype == object:
    y = df['sentiment'].map({'negative':0,'positive':1}).values
else:
    y = df['sentiment'].values

print("X_glove shape:", X_glove.shape)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X_glove, y, test_size=0.2, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, digits=4))
df.head()

In [ ]:
#ANN
label_col = "sentiment"
le = LabelEncoder()
y = le.fit_transform(df[label_col].astype(str))

X_train_tfidf, X_val_tfidf, y_train, y_val = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

# convert sparse tfidf → dense
X_train = X_train_tfidf.toarray()
X_val = X_val_tfidf.toarray()

input_dim = X_train.shape[1]

model = Sequential([
    Dense(64, activation="relu", input_shape=(input_dim,)),
    Dropout(0.3),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=5, batch_size=128)


model.save(os.path.join(OUT_DIR, "tfidf_ann_model.keras"))
print("Model saved successfully!")


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_col = "sentiment"   # name of the column in df
le = LabelEncoder()

y = le.fit_transform(df[label_col].astype(str))


In [ ]:
print(type(X_tfidf))
print(type(y))
print(X_tfidf.shape)
print(len(y))


In [ ]:
# ================================
# GENETIC ALGORITHM FOR ANN TUNING
# ================================

import numpy as np
import random
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Make sure you already have:
# X_tfidf (TF-IDF features) and y (labels) defined from your previous code.

# 1. Train-validation split for GA (separate from previous if you want)
GA_X_train, GA_X_val, GA_y_train, GA_y_val = train_test_split(
    X_tfidf.toarray(),  # convert sparse -> dense
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

input_dim = GA_X_train.shape[1]

# 2. Define hyperparameter search space
NEURONS_1_RANGE = (32, 256)   # first hidden layer neurons
NEURONS_2_RANGE = (8, 128)    # second hidden layer neurons
DROPOUT_RANGE   = (0.1, 0.5)  # dropout rate
LR_EXP_RANGE    = (-4, -2)    # learning rate = 10^exp -> between 1e-4 and 1e-2

POP_SIZE    = 10   # population size
N_GENERATIONS = 5  # number of generations
MUTATION_RATE = 0.3

# 3. Helper to create a random individual (chromosome)
def create_individual():
    n1 = random.randint(NEURONS_1_RANGE[0] // 16, NEURONS_1_RANGE[1] // 16) * 16
    n2 = random.randint(NEURONS_2_RANGE[0] // 8, NEURONS_2_RANGE[1] // 8) * 8
    dr = round(random.uniform(DROPOUT_RANGE[0], DROPOUT_RANGE[1]), 2)
    lr_exp = random.uniform(LR_EXP_RANGE[0], LR_EXP_RANGE[1])
    return {
        "n1": n1,
        "n2": n2,
        "dropout": dr,
        "lr_exp": lr_exp  # learning rate = 10 ** lr_exp
    }

# 4. Build ANN model from an individual
def build_model(individual, input_dim):
    tf.keras.backend.clear_session()
    lr = 10 ** individual["lr_exp"]

    model = Sequential([
        Dense(individual["n1"], activation="relu", input_shape=(input_dim,)),
        Dropout(individual["dropout"]),
        Dense(individual["n2"], activation="relu"),
        Dense(1, activation="sigmoid")
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=optimizer,
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

# 5. Fitness function: validation accuracy after few epochs
def fitness(individual):
    model = build_model(individual, input_dim)
    history = model.fit(
        GA_X_train, GA_y_train,
        epochs=3,             # small number for speed
        batch_size=128,
        validation_data=(GA_X_val, GA_y_val),
        verbose=0
    )
    val_acc = max(history.history["val_accuracy"])
    return val_acc

# 6. Selection: pick top-k by fitness
def select_elites(population, fitnesses, k=4):
    # sort by fitness descending
    sorted_indices = np.argsort(fitnesses)[::-1]
    elites = [population[i] for i in sorted_indices[:k]]
    elite_fits = [fitnesses[i] for i in sorted_indices[:k]]
    return elites, elite_fits

# 7. Crossover: combine two parents
def crossover(parent1, parent2):
    child = {}
    for key in parent1.keys():
        child[key] = random.choice([parent1[key], parent2[key]])
    return child

# 8. Mutation: randomly tweak genes
def mutate(individual):
    ind = individual.copy()

    if random.random() < MUTATION_RATE:
        ind["n1"] = random.randint(NEURONS_1_RANGE[0] // 16, NEURONS_1_RANGE[1] // 16) * 16
    if random.random() < MUTATION_RATE:
        ind["n2"] = random.randint(NEURONS_2_RANGE[0] // 8, NEURONS_2_RANGE[1] // 8) * 8
    if random.random() < MUTATION_RATE:
        ind["dropout"] = round(random.uniform(DROPOUT_RANGE[0], DROPOUT_RANGE[1]), 2)
    if random.random() < MUTATION_RATE:
        ind["lr_exp"] = random.uniform(LR_EXP_RANGE[0], LR_EXP_RANGE[1])

    return ind

# 9. Initialize population
population = [create_individual() for _ in range(POP_SIZE)]

# 10. Run GA loop
best_overall = None
best_fitness_overall = 0.0

for gen in range(N_GENERATIONS):
    print(f"\n=== Generation {gen+1}/{N_GENERATIONS} ===")

    # evaluate fitness of each individual
    fitnesses = []
    for idx, ind in enumerate(population):
        print(f"  Evaluating individual {idx+1}/{len(population)}: {ind}")
        fit = fitness(ind)
        fitnesses.append(fit)
        print(f"    -> val_accuracy = {fit:.4f}")

    # selection
    elites, elite_fits = select_elites(population, fitnesses, k=4)
    print("  Best in this generation:", elites[0], "fitness =", elite_fits[0])

    # update global best
    if elite_fits[0] > best_fitness_overall:
        best_fitness_overall = elite_fits[0]
        best_overall = elites[0].copy()

    # create new population
    new_population = elites.copy()

    # generate children until population size is reached
    while len(new_population) < POP_SIZE:
        parent1, parent2 = random.sample(elites, 2)
        child = crossover(parent1, parent2)
        child = mutate(child)
        new_population.append(child)

    population = new_population

print("\n======================")
print("Best hyperparameters found by GA:")
print(best_overall)
print("Best validation accuracy:", best_fitness_overall)

# 11. Train final model with best hyperparameters
print("\nTraining final ANN model with best GA hyperparameters...")
final_model = build_model(best_overall, input_dim)
history = final_model.fit(
    GA_X_train, GA_y_train,
    epochs=5,          # more epochs for final model
    batch_size=128,
    validation_data=(GA_X_val, GA_y_val),
    verbose=1
)

# Evaluate on validation set
val_pred = (final_model.predict(GA_X_val) > 0.5).astype(int).flatten()
print("Final validation accuracy:", accuracy_score(GA_y_val, val_pred))

# Save final GA-optimized model
ga_model_path = os.path.join(OUT_DIR, "tfidf_ann_ga_optimized.keras")
final_model.save(ga_model_path)
print("GA-optimized model saved at:", ga_model_path)
